In [ ]:
import torch
from türkçe_dil_modelim.tokenization.tokenizer import Tokenizer
from türkçe_dil_modelim.model import Model

tokenizer= Tokenizer("türkçe_dil_modelim/tokenizer.json")
prompt="Benim adım Burak"
prompts= [
    "Benim adım Burak",
    "senin adın ",
    "nerede yaşıyorsun"
]# eğer batch batch eğitmek istiyorsam bu şekilde vermem gerekiyo verileri.
#mesela resim verisinde bu durum her bir resim özelinde 2D bir liste. cümle için ise 1d bir liste oluyor
tokens = tokenizer.encode(prompts[0])
tokens.to("cpu")


tensor([1179,  372, 1148,  353,  622,  373,  342, 1152,  622, 1179,  323,  373,
        1153])

In [2]:
batch_tokens= tokenizer.encode_batch(prompts,32)

In [3]:
batch_tokens.shape

torch.Size([3, 32])

In [4]:
batch_tokens

tensor([[1179,  372, 1148,  353,  622,  373,  342, 1152,  622, 1179,  323,  373,
         1153,  627,  627,  627,  627,  627,  627,  627,  627,  627,  627,  627,
          627,  627,  627,  627,  627,  627,  627,  627],
        [ 497,  622,  373,  342, 1148,  627,  627,  627,  627,  627,  627,  627,
          627,  627,  627,  627,  627,  627,  627,  627,  627,  627,  627,  627,
          627,  627,  627,  627,  627,  627,  627,  627],
        [ 534, 1162,  462,  622,  371,  297,  425,  627,  627,  627,  627,  627,
          627,  627,  627,  627,  627,  627,  627,  627,  627,  627,  627,  627,
          627,  627,  627,  627,  627,  627,  627,  627]])

In [8]:
context_length= 32
torch.manual_seed(1)

model= Model(vocab_size=tokenizer.get_vocab_size(),embedding_dim=12,num_heads=4,context_length=context_length,num_layers=8,device="cpu")

In [9]:
out= model(batch_tokens)
out=out.squeeze(0)
out.shape

torch.Size([3, 32, 2881])

In [10]:
out=model.generate(tokens,2, temperature=10,top_k=12,top_p=0.4)
tokenizer.decode(out)

'Benim adım Burakgeriı'

In [11]:
token_ids= tokenizer.encode(text)
len(token_ids), type(token_ids)

(693, torch.Tensor)

In [12]:
from türkçe_dil_modelim.text_dataset import create_data_loader

stride=12

train_data_loader= create_data_loader(token_ids=token_ids.tolist(),context_length=context_length,stride=stride,batch_size=3,shuffle=True)

In [13]:
parameters_count= sum(p.numel() for p in model.parameters())
print(parameters_count)

print(model)

55712
Model(
  (embedding): Embedding(
    (embedding): Embedding(1376, 12)
  )
  (layers): Sequential(
    (0): DecoderBlock(
      (attention): MultiHeadAttention(
        (multi_head_attention): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=12, out_features=12, bias=True)
        )
        (projection): Linear(in_features=12, out_features=12, bias=True)
      )
      (layer_norm1): LayerNorm()
      (layer_norm2): LayerNorm()
      (mlp): MLP(
        (gate_proj): Linear(in_features=12, out_features=48, bias=True)
        (up_proj): Linear(in_features=12, out_features=48, bias=True)
        (down_proj): Linear(in_features=48, out_features=12, bias=True)
        (gelu): GELU()
      )
    )
    (1): DecoderBlock(
      (attention): MultiHeadAttention(
        (multi_head_attention): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=12, out_features=12, bias=True)
        )
        (projection): Linear(in_feat

In [14]:
out= model(batch_tokens)
out,out.flatten(),out.flatten(0,1),out.flatten(0,1).shape

(tensor([[[ 0.2742,  0.5289, -0.0548,  ..., -0.0746, -0.6976,  0.2787],
          [ 0.0202, -0.4472,  0.9803,  ..., -0.8815,  0.1148, -1.4019],
          [-0.2123, -0.1348,  0.3730,  ..., -0.6394, -0.9094, -0.5363],
          ...,
          [-0.0138,  0.5016,  0.3422,  ..., -0.6926,  0.2629,  0.8766],
          [-0.2290,  0.2922,  0.2491,  ..., -0.4324,  0.5385,  0.9477],
          [-0.1798,  0.0876,  0.2358,  ..., -0.2111,  0.7711,  0.7074]],
 
         [[ 0.2178,  0.1661, -0.2212,  ..., -0.0480, -0.2023, -0.2502],
          [ 0.2254,  0.2809,  0.3346,  ...,  0.0655,  1.3672,  0.4926],
          [-0.5262,  0.3214,  0.1533,  ..., -0.6147,  0.4229,  1.1262],
          ...,
          [-0.0138,  0.5016,  0.3422,  ..., -0.6926,  0.2629,  0.8766],
          [-0.2290,  0.2922,  0.2491,  ..., -0.4324,  0.5385,  0.9477],
          [-0.1798,  0.0876,  0.2358,  ..., -0.2111,  0.7711,  0.7074]],
 
         [[-0.4560, -0.6097,  0.4711,  ..., -0.6506,  1.2569, -0.4507],
          [-0.8877, -0.1470,

In [15]:
for i , (X,Y) in enumerate(train_data_loader):
    print(X.shape,Y.shape,Y.flatten().shape)
    break

torch.Size([3, 32]) torch.Size([3, 32]) torch.Size([96])


In [ ]:
import torch.nn as nn 
device="cpu"
loss_fn= nn.CrossEntropyLoss()

optimizer= torch.optim.AdamW(model.parameters(),lr=1e-3)

epochs= 20

for epoch in range(epochs):
    total_loss=0
    for i, (X,Y) in enumerate(train_data_loader):
        X= X.to(device)
        Y= Y.to(device)

        pred= model(X)
        loss= loss_fn(pred.flatten(0,1),Y.flatten())
        total_loss += loss.item()

        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
    
    average_loss= total_loss/ len(train_data_loader)
    print(f"Epoch {epoch+1} loss: {loss.item()} average_loss: {average_loss}")

In [ ]:
#temperature , top_k , top_p


In [ ]:

import torch

# out bir int veya long tensor ise önce float tensor yap
out_tensor = torch.tensor([out], dtype=torch.float32)

probs = torch.softmax(out_tensor, dim=-1)

_, max_index = torch.max(probs, dim=-1)
next_token = max_index.item()

In [ ]:
probs.squeeze(0) # her zaman en yüksek olasılıklıyı seçmesin diye temperature, top_k, top_p gibi parametreleri de kullanıcaz

tensor([5.0000e-01, 0.0000e+00, 1.7212e-14, 0.0000e+00, 0.0000e+00, 0.0000e+00,
        0.0000e+00, 9.3976e-13, 0.0000e+00, 5.0000e-01, 0.0000e+00, 0.0000e+00,
        2.5545e-12, 0.0000e+00, 0.0000e+00])

In [ ]:
next_token

0

In [ ]:
probs= probs.squeeze(0)

In [ ]:
probs

tensor([5.0000e-01, 0.0000e+00, 1.7212e-14, 0.0000e+00, 0.0000e+00, 0.0000e+00,
        0.0000e+00, 9.3976e-13, 0.0000e+00, 5.0000e-01, 0.0000e+00, 0.0000e+00,
        2.5545e-12, 0.0000e+00, 0.0000e+00])

In [ ]:
import torch

# 3 token için olasılık dağılımı
probs = torch.tensor([0.1, 0.3, 0.6])

# 1 token seç
sample = torch.multinomial(probs, 1)


counts= [0,0,0]
for i in range(100000): # bu kısımda deneme sayısı arttıkça gerçek olasılıklarıyla aynı değer olmaya yakınlaşacak çıktı
    sample= torch.multinomial(probs,1).item()
    counts[sample]+=1
print(counts)

[9964, 30021, 60015]


[[[1179,
   372,
   1148,
   353,
   622,
   373,
   342,
   1152,
   622,
   1179,
   323,
   373,
   1153,
   179,
   305]]]

In [ ]:
out_tensor

tensor([[1179.,  372., 1148.,  353.,  622.,  373.,  342., 1152.,  622., 1179.,
          323.,  373., 1153.,  179.,  305.]])

In [ ]:

import torch

# out bir int veya long tensor ise önce float tensor yap
out_tensor = torch.tensor([out], dtype=torch.float32)

probs = torch.softmax(out_tensor/1.5, dim=-1) # temperature denemesi

probs= probs.squeeze(0)

sample_count={}
for _ in range(10000):
    sample= torch.multinomial(probs,1)
    sample_count[sample.item()]= sample_count.get(sample.item(),0)+1
sample_count

{9: 4948, 0: 5052}

In [ ]:
probs

tensor([5.0000e-01, 0.0000e+00, 1.7212e-14, 0.0000e+00, 0.0000e+00, 0.0000e+00,
        0.0000e+00, 9.3976e-13, 0.0000e+00, 5.0000e-01, 0.0000e+00, 0.0000e+00,
        2.5545e-12, 0.0000e+00, 0.0000e+00])

In [ ]:
temperature=0.1 # temperature değeri 1'den büyük olduğunda olasılıklar birbirine yaklaştığından modelin farklı eyler deme olasılığı artar
#1'den küçükse model daha determenistik olur
out_tensor = torch.tensor([out], dtype=torch.float32)

adjusted_out= out_tensor/temperature
torch.softmax(adjusted_out,dim=-1)

tensor([[0.5000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.5000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000]])

In [ ]:
outs= {}

for _ in range (100):
    response= model.generate(tokens,max_new_tokens=2,temperature=0.4)
    decode= tokenizer.decode(response)
    outs[decode]= outs.get(decode,0) +1

outs


{'Benim adım Burakbotanikakısa': 1,
 'Benim adım Burakaşçıfibroyad': 1,
 'Benim adım Burakhikingçarşamba': 1,
 'Benim adım Burakkırıküyorlar': 1,
 'Benim adım BurakfiltirlemekTennis': 1,
 'Benim adım Buraküyorumcuma': 1,
 'Benim adım Burakdamatavukat': 1,
 'Benim adım BurakAmazonparlak': 1,
 'Benim adım Buraktemizlemekooforit': 1,
 'Benim adım Burakcı': 1,
 'Benim adım Burakflüoresanlaşmaküyle': 1,
 'Benim adım Burakbahçeiğne': 1,
 'Benim adım Burakkapıçözelmek': 1,
 'Benim adım Buraküyorumsarcoma': 1,
 'Benim adım Burakmicroservicesgüzel': 1,
 'Benim adım Burak.var': 1,
 'Benim adım Buraketgöstermek': 1,
 'Benim adım BurakamcaHybrid': 1,
 'Benim adım Burakuyorsunuzmi': 1,
 'Benim adım Burakşarkıaids': 1,
 'Benim adım Burakneredenkoyu': 1,
 'Benim adım BurakyaniParkinson': 1,
 'Benim adım BurakkalkamkAmerica': 1,
 'Benim adım Burakyanmaokul': 1,
 'Benim adım Buraksmsh': 1,
 'Benim adım Burakheykeltıraşlıkkilit': 1,
 'Benim adım Burakhikinghayvancılık': 1,
 'Benim adım Burakaşçısebze': 

In [ ]:
out_tensor

tensor([1179.,  372., 1148.,  353.,  622.,  373.,  342., 1152.,  622., 1179.,
         323.,  373., 1153.,  311.,   59.])

In [ ]:
sorted_outs= sorted(out_tensor.tolist(), reverse=True)
sorted_indexes=[]
top_k=10
temperature= 0.5
for so in sorted_outs[:top_k]:
    so_index= out_tensor.tolist().index(so)
    sorted_indexes.append(so_index)
sorted_outs= torch.tensor(sorted_outs[:top_k])
adjusted_out= torch.softmax(sorted_outs/temperature,dim=-1)
sorted_outs, sorted_indexes, adjusted_out

(tensor([1179., 1179., 1153., 1152., 1148.,  622.,  622.,  373.,  373.,  372.]),
 [0, 0, 12, 7, 2, 4, 4, 5, 5, 1],
 tensor([5.0000e-01, 5.0000e-01, 1.3051e-23, 1.7663e-24, 5.9253e-28, 0.0000e+00,
         0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00]))

In [ ]:
def top_p(self,logits,top_p):
    sorted_logits, sorted_indices= torch.sort(logits,descending=-1)
    cumulative_probs= torch.cumsum(torch.softmax(sorted_logits,dim=-1),dim=-1)
    r= cumulative_probs>top_p
    r[...,1:]=r[...,:-1].clone()
    r[...,0]=False

    sorted_logits[r]=-float('inf') 
    filtered_logits= torch.full_like(logits,-float('inf'))

    filtered_logits.scatter_(0,sorted_indices,sorted_logits)
    return filtered_logits



In [ ]:
import torch
a= torch.tensor([[1.0,2.0,3.0],[1.0,5.0,3.0],[7.0,2.0,3.0]]) #logit
#topk

temp=1
a/=temp

b=torch.softmax(a,dim=-1)#top

b


tensor([[0.0900, 0.2447, 0.6652],
        [0.0159, 0.8668, 0.1173],
        [0.9756, 0.0066, 0.0179]])

In [ ]:
import torch
from türkçe_dil_modelim.tokenization.tokenizer import Tokenizer

# Tokenizer'ı yükle
tokenizer = Tokenizer(
    vocab_file="/workspaces/llm-from-scratch/türkçe_dil_modelim/tokenizer.json",
    auto_learn=True
)

print(f"Başlangıç vocab boyutu: {tokenizer.get_vocab_size()}")


wiki_file = "/workspaces/llm-from-scratch/data/wikipedia_tr_5m_words.txt"
batch_size = 1000  
context_length = 1000  


def process_wikipedia(file_path, tokenizer, batch_size=1000, context_length=50):
    total_lines = 0
    updated_vocab = tokenizer.get_vocab_size()
    
    with open(file_path, "r", encoding="utf-8") as f:
        batch_texts = []
        for line in f:
            line = line.strip()
            if line:  # boş satırları atla
                batch_texts.append(line)
                total_lines += 1

            if len(batch_texts) == batch_size:
                # batch encode
                encoded = tokenizer.encode_batch(batch_texts, context_length)
                # opsiyonel: batch decode ile kontrol
                # decoded = [tokenizer.decode(seq) for seq in encoded]
                batch_texts = []

                # her batch sonunda vocab büyüklüğünü yazdır
                new_vocab = tokenizer.get_vocab_size()
                if new_vocab != updated_vocab:
                    print(f"{total_lines} satır işlendi, vocab büyüklüğü: {new_vocab}")
                    updated_vocab = new_vocab

        # Son batchi işle
        if batch_texts:
            encoded = tokenizer.encode_batch(batch_texts, context_length)
            new_vocab = tokenizer.get_vocab_size()
            if new_vocab != updated_vocab:
                print(f"{total_lines} satır işlendi, vocab büyüklüğü: {new_vocab}")

    # İşlem bitince vocab'ı kaydet
    tokenizer.save_vocab()
    print(f"Tüm veri işlendi. Final vocab boyutu: {tokenizer.get_vocab_size()}")


process_wikipedia(wiki_file, tokenizer, batch_size=batch_size, context_length=context_length)


Başlangıç vocab boyutu: 1377
1000 satır işlendi, vocab büyüklüğü: 1998
2000 satır işlendi, vocab büyüklüğü: 2470
3000 satır işlendi, vocab büyüklüğü: 2795
3384 satır işlendi, vocab büyüklüğü: 2876
Tüm veri işlendi. Final vocab boyutu: 2876


In [ ]:
import torch
from türkçe_dil_modelim.tokenization.tokenizer import Tokenizer
from türkçe_dil_modelim.text_dataset import TextDataset
from torch.utils.data import DataLoader
from pathlib import Path



tokenizer = Tokenizer(
    "türkçe_dil_modelim/tokenizer.json",
    auto_learn=False
)
wiki_file = "/workspaces/llm-from-scratch/data/wikipedia_encoded.txt"

batch_files = TextDataset.process_and_save_batches(
    file_path=wiki_file,
    tokenizer=tokenizer,
    context_length=32,
    stride=1,
    batch_size=64,
    save_dir="wikipedia_batches"
)

for batch_file in batch_files[:2]:  # örnek olarak ilk 2 batch
    batch_data = torch.load(batch_file)
    x, y = batch_data["input"], batch_data["target"]
    print("Input shape:", x.shape)
    print("Target shape:", y.shape)


5000 satır işlendi, vocab boyutu: 2875


: 

In [1]:
import re

input_file = "/workspaces/llm-from-scratch/data/wikipedia_tr_5m_words.txt"
output_file = "/workspaces/llm-from-scratch/data/wikipedia_formatted.txt"

with open(input_file, "r", encoding="utf-8") as f:
    text = f.read()

# Küçük harfe çevir (istersen kaldırabilirsin)
text = text.lower()

# Fazla boşlukları temizle
text = re.sub(r"\s+", " ", text)

# Cümlelere böl (., !, ?)
sentences = re.split(r'(?<=[.!?])\s+', text)

clean_sentences = []

for sentence in sentences:
    sentence = sentence.strip()

    # Çok kısa ve anlamsız parçaları at
    if len(sentence) < 10:
        continue

    # Cümle sonundaki nokta vs kalsın
    clean_sentences.append(f"<başla> {sentence} <bitiş>")

with open(output_file, "w", encoding="utf-8") as f:
    for s in clean_sentences:
        f.write(s + "\n")

print("Formatlama tamamlandı.")
print("Toplam cümle:", len(clean_sentences))


Formatlama tamamlandı.
Toplam cümle: 326623


In [ ]:
from türkçe_dil_modelim.tokenization.tokenizer import Tokenizer

tokenizer = Tokenizer("/workspaces/llm-from-scratch/türkçe_dil_modelim/tokenizer.json",auto_learn=False)

input_file = "/workspaces/llm-from-scratch/data/wikipedia_formatted.txt"
output_file = "/workspaces/llm-from-scratch/data/wikipedia_encode.txt"

with open(input_file,"r",encoding="utf-8") as f:
    lines= f.readlines()

with open(output_file,"w",encoding="utf-8") as out:
   
    for line in lines:
        token_ids=tokenizer.encode(line)
        token_ids_str = " ".join(map(str, token_ids.tolist()))
        out.write(token_ids_str + "\n")

tokenizer.save_vocab()

print("Encode işlemi tamamlandı.")


/usr/local/python/3.12.1/lib/python3.12/site-packages/torch/_subclasses/functional_tensor.py:283: UserWarning: Failed to initialize NumPy: No module named 'numpy' (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:84.)
  cpu = _conversion_method_template(device=torch.device("cpu"))


Encode işlemi tamamlandı.


In [ ]:
from türkçe_dil_modelim.model import Model
import torch
from türkçe_dil_modelim.tokenization.tokenizer import Tokenizer
from türkçe_dil_modelim.model import Model


tokenizer= Tokenizer("türkçe_dil_modelim/tokenizer.json")

model = Model(
    vocab_size=tokenizer.get_vocab_size(),
    embedding_dim=128,
    num_heads=4,
    context_length=128,
    num_layers=4,
    device="cpu"
)
sum([p.numel() for p in model.parameters()])

/usr/local/python/3.12.1/lib/python3.12/site-packages/torch/_subclasses/functional_tensor.py:283: UserWarning: Failed to initialize NumPy: No module named 'numpy' (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:84.)
  cpu = _conversion_method_template(device=torch.device("cpu"))


1476960

In [2]:
from torch.utils.data import DataLoader
from türkçe_dil_modelim.trainer import Trainer
from türkçe_dil_modelim.text_dataset import TextDataset

token_ids = TextDataset.load_encoded_file("data/wikipedia_encode.txt")
small_token_ids = token_ids[:200_000]  # ilk 200k token

dataset = TextDataset(small_token_ids, context_length=128, stride=64)
dataloader = DataLoader(dataset, batch_size=64, shuffle=True)

print("Dataset size:", len(dataset))

trainer = Trainer(
    model=model,
    dataloader=dataloader,
    device="cpu",
    lr=2e-4,
    epochs=5,
    save_every=2000
)

trainer.train()


FileNotFoundError: [Errno 2] No such file or directory: 'data/wikipedia_encode.txt'

In [1]:
!pip install transformers


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 56.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 553.3/553.3 kB 36.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 69.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 83.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 80.3 MB/s  0:00:006m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.6/803.6 kB 50.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16/16 [transformers] [transformers]ub]

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python3 -m pip install --upgrade pip


In [1]:
!pip install sentencepiece

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 39.2 MB/s  0:00:00

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python3 -m pip install --upgrade pip


In [ ]:
from türkçe_dil_modelim.bpetokenizer import TurkishBPETokenizer
tokenizer = TurkishBPETokenizer(vocab_size=32000)

tokenizer.train("/Users/burak.yilmaz/Desktop/kendi_dil_modelim/data/wikipedia_formatted.txt")

tokenizer.save_tokenizer_json("tokenizer.json")


In [ ]:
from türkçe_dil_modelim.model import Model
import torch
from türkçe_dil_modelim.tokenization.tokenizer import Tokenizer
from türkçe_dil_modelim.model import Model


tokenizer= Tokenizer("türkçe_dil_modelim/tokenizer.json")

model = Model(
    vocab_size=tokenizer.get_vocab_size(),
    embedding_dim=256,
    num_heads=4,
    context_length=256,
    num_layers=4,
    device="cpu"
)
sum([p.numel() for p in model.parameters()]), tokenizer.get_vocab_size()


from pathlib import Path
import random

batch_dir = Path("wikipedia_batches")
all_batches = list(batch_dir.glob("*.pt"))
all_batches.sort()  # deterministik sıra için

random.seed(42)  # reproducibility
random.shuffle(all_batches)

n = len(all_batches)
train_files = all_batches[: int(n*0.8)]
val_files   = all_batches[int(n*0.8): int(n*0.9)]
test_files  = all_batches[int(n*0.9):]

print(f"Train: {len(train_files)}, Val: {len(val_files)}, Test: {len(test_files)}")

from torch.utils.data import DataLoader, TensorDataset
import torch

def batch_files_to_loader(batch_files, batch_size=64, device="cpu"):
    datasets = []
    for file in batch_files:
        batch = torch.load(file)
        x = batch["input"].to(device)
        y = batch["target"].to(device)
        datasets.append(TensorDataset(x, y))
    # Tüm batch’leri birleştir
    full_dataset = torch.utils.data.ConcatDataset(datasets)
    loader = DataLoader(full_dataset, batch_size=batch_size, shuffle=True)
    return loader

device = "cuda" if torch.cuda.is_available() else "cpu"

train_loader = batch_files_to_loader(train_files, batch_size=64, device=device)
val_loader   = batch_files_to_loader(val_files, batch_size=64, device=device)
test_loader  = batch_files_to_loader(test_files, batch_size=64, device=device)

from türkçe_dil_modelim.trainer import Trainer
trainer = Trainer(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    test_loader=test_loader,
    device=device,
    lr=3e-4,
    weight_decay=0.01,
    epochs=20,
    grad_clip=1.0,
    save_dir="training_outputs"
)
trainer.train()

In [ ]:
from türkçe_dil_modelim.text_dataset import TextDataset
from türkçe_dil_modelim.bpetokenizer import TurkishBPETokenizer
from türkçe_dil_modelim.trainer import Trainer
from torch.utils.data import DataLoader


token_ids = []
tokenizer= TurkishBPETokenizer("/workspaces/llm-from-scratch/türkçe_dil_modelim/tokenizer_a.json")

with open("wikipedia.txt", "r", encoding="utf-8") as f:
    for line in f:
        token_ids.extend(tokenizer.encode(line))


context_length = 256
stride = 128
batch_size = 64

dataset = TextDataset(token_ids, context_length, stride)
loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)




device = "cuda" if torch.cuda.is_available() else "cpu"

trainer = Trainer(
    model=model,
    train_loader=loader,  # istersen train_loader, val_loader, test_loader ayrılabilir
    val_loader=loader,
    test_loader=loader,
    device=device,
    epochs=20
)

trainer.train()

In [ ]:
from türkçe_dil_modelim.bpetokenizer import TurkishBPETokenizer
import json

with open("türkçe_dil_modelim/tokenizer.json", "r", encoding="utf-8") as f:
    data = json.load(f)

tokenizer = TurkishBPETokenizer(vocab_size=data["model"]["vocab_size"])
tokenizer.vocab = data["model"]["vocab"]
tokenizer.reverse_vocab = {int(i): tok for tok, i in tokenizer.vocab.items()}

tokenizer.unk_id = tokenizer.vocab["<unk>"]
tokenizer.pad_id = tokenizer.vocab["<pad>"]
tokenizer.bos_id = tokenizer.vocab["<bos>"]
tokenizer.eos_id = tokenizer.vocab["<eos>"]

tokenizer.merges = [tuple(m.split(" ")) for m in data["model"].get("merges", [])]

In [1]:
from türkçe_dil_modelim.bpetokenizer import TurkishBPETokenizer

tokenizer= TurkishBPETokenizer()
tokenizer.load_tokenizer_json("/workspaces/llm-from-scratch/türkçe_dil_modelim/tokenizer.json")
tokenizer.encode("Merhaba Dünya",add_special_tokens=True)

/usr/local/python/3.12.1/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Tokenizer başarıyla yüklendi.


[2, 9082, 5786, 1105, 4032, 3]

In [2]:
# Multi-Head Attention'da embedding_dim, num_heads'e tam bölünmelidir.
# Çünkü her head için boyut şu şekilde hesaplanır:
#     head_dim = embedding_dim // num_heads
#
# Model içinde tensor reshape işlemi yapılır:
#     [batch, seq_len, embedding_dim]
#         → [batch, seq_len, num_heads, head_dim]
#
# Eğer embedding_dim, num_heads'e tam bölünmezse bu reshape mümkün olmaz.
#
# Genelde head_dim 32 veya 64 seçilir:
# - Çok küçük head_dim → düşük temsil gücü
# - Çok büyük head_dim → az head, düşük paralellik
# - 32 gibi değerler donanım (GPU/MPS) için de daha verimlidir
#
# Örnek doğru kombinasyonlar:
# 128 / 4 = 32
# 192 / 6 = 32
# 160 / 5 = 32

In [ ]:
from türkçe_dil_modelim.text_dataset import TextDataset
import torch
from torch.utils.data import random_split
from torch.utils.data import DataLoader


context_length = 128
stride = 32
token_tensor = torch.load(
    "/Users/burak.yilmaz/Desktop/kendi_dil_modelim/encoded_wikipedia.pt"
)
print("Toplam token:", len(token_tensor))
full_dataset = TextDataset(
    token_tensor,
    context_length=context_length,
    stride=stride
)

print("Toplam sample:", len(full_dataset))


train_size = int(0.9 * len(full_dataset))
val_size = int(0.05 * len(full_dataset))
test_size = len(full_dataset) - train_size - val_size

train_dataset, val_dataset, test_dataset = random_split(
    full_dataset,
    [train_size, val_size, test_size]
)

print("Train:", len(train_dataset))
print("Val:", len(val_dataset))
print("Test:", len(test_dataset))



batch_size = 16  # M1 için güvenli

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size
)

from türkçe_dil_modelim.model import Model
from türkçe_dil_modelim.bpetokenizer import TurkishBPETokenizer

tokenizer = TurkishBPETokenizer()
tokenizer.load_tokenizer_json("/Users/burak.yilmaz/Desktop/kendi_dil_modelim/türkçe_dil_modelim/tokenizer.json")


model = Model(
    vocab_size=len(tokenizer.vocab),
    embedding_dim=160,
    num_heads=5,
    context_length=128,
    num_layers=5,
    device="cpu"
)

print("Parametre sayısı:",
      sum(p.numel() for p in model.parameters()))


from türkçe_dil_modelim.trainer import Trainer  # güncel Trainer dosyan

trainer = Trainer(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    test_loader=test_loader,
    device="cpu",
    lr=3e-4,
    epochs=5,              # İlk deneme için 5
    warmup_steps=500,
    grad_clip=1.0
)


trainer.train()

In [1]:
from türkçe_dil_modelim.bpetokenizer import TurkishBPETokenizer
from türkçe_dil_modelim.model import Model
import torch


tokenizer= TurkishBPETokenizer()
tokenizer.load_tokenizer_json("/workspaces/llm-from-scratch/türkçe_dil_modelim/tokenizer.json")

VOCAB_SIZE= len(tokenizer.vocab)

model= Model(
    vocab_size=VOCAB_SIZE,
    embedding_dim=160,
    num_heads=5,
    context_length=128,
    num_layers=5,
    device="cpu"
)

checkpoint= torch.load("/workspaces/llm-from-scratch/türkçe_dil_modelim/best_model.pt",map_location="cpu")

if "model_state_dict" in checkpoint:
    checkpoint= checkpoint["model_state_dict"]

model.load_state_dict(checkpoint)
model.eval()

print("model yüklendi")

prompt = "rivayetlere ve efsaneye göre "
input_ids = tokenizer.encode(prompt)

input_ids = torch.tensor(input_ids, dtype=torch.long)

output_ids = model.generate(
    input_ids,
    max_new_tokens=25,
    temperature=0.8,
    top_k=20,
    top_p=0.9
)

generated_text = tokenizer.decode(output_ids.tolist())

print("\n OUTPUT:\n")
print(generated_text)




/usr/local/python/3.12.1/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Tokenizer başarıyla yüklendi.
model yüklendi

 OUTPUT:

merkez ilçede en eski millî futbolcu 1984 - paul fıstığı ve bu nedenle bu konuda çok önemli bir şey ifade edilir. bir süre


In [1]:
from türkçe_dil_modelim.bpetokenizer import TurkishBPETokenizer
from türkçe_dil_modelim.model import Model
import torch


tokenizer= TurkishBPETokenizer()
tokenizer.load_tokenizer_json("/workspaces/llm-from-scratch/türkçe_dil_modelim/tokenizer.json")

VOCAB_SIZE= len(tokenizer.vocab)

model= Model(
    vocab_size=VOCAB_SIZE,
    embedding_dim=160,
    num_heads=5,
    context_length=128,
    num_layers=5,
    device="cpu"
)

model

Tokenizer başarıyla yüklendi.


Model(
  (embedding): Embedding(
    (embedding): Embedding(15985, 160)
  )
  (layers): Sequential(
    (0): DecoderBlock(
      (attention): MultiHeadAttention(
        (multi_head_attention): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=160, out_features=160, bias=True)
        )
        (projection): Linear(in_features=160, out_features=160, bias=True)
      )
      (layer_norm1): LayerNorm()
      (layer_norm2): LayerNorm()
      (mlp): MLP(
        (gate_proj): Linear(in_features=160, out_features=640, bias=True)
        (up_proj): Linear(in_features=160, out_features=640, bias=True)
        (down_proj): Linear(in_features=640, out_features=160, bias=True)
        (gelu): GELU()
      )
    )
    (1): DecoderBlock(
      (attention): MultiHeadAttention(
        (multi_head_attention): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=160, out_features=160, bias=True)
        )
        (projection): Linear

In [8]:
from türkçe_dil_modelim.bpetokenizer import TurkishBPETokenizer
tokenizer= TurkishBPETokenizer()
tokenizer.load_tokenizer_json("/workspaces/llm-from-scratch/türkçe_dil_modelim/tokenizer.json")

text= "evlerimiz"
ids=tokenizer.encode(text)


print(f"Ortalama token/kelime oranı: {len(ids)/len(text.split()):.3f}") 

Tokenizer başarıyla yüklendi.
Ortalama token/kelime oranı: 3.000


In [9]:
text = "Evlerimiz çok güzeldi ve arkadaşlarımızla oynadık."
ids = tokenizer.encode(text)

print("Toplam token:", len(ids))
print("Toplam kelime:", len(text.split()))
print("Token/Kelime:", len(ids)/len(text.split()))

Toplam token: 15
Toplam kelime: 6
Token/Kelime: 2.5


In [ ]:
from türkçe_dil_modelim.bpetokenizer import TurkishBPETokenizer
from türkçe_dil_modelim.model import Model
import torch

tokenizer= TurkishBPETokenizer()
tokenizer.load_tokenizer_json("/workspaces/llm-from-scratch/türkçe_dil_modelim/tokenizer.json")

model= Model(
    vocab_size=len(tokenizer.vocab),
    embedding_dim=384,
    num_heads=12,
    context_length=256,
    num_layers=8,
    device="cpu"
)
print("Model parametre sayısı:", sum(p.numel() for p in model.parameters()))


tokens_chunk= []
file_path= "/workspaces/llm-from-scratch/data/wikipedia_formatted.txt"
with open(file_path, "r", encoding="utf-8") as f:
    batch_lines = []

    for i,line in enumerate(f):
        line = line.strip()
        if not line:
            continue
        
        ids = tokenizer.encode(line,add_special_tokens=True)
        tokens_chunk.append(torch.tensor(ids, dtype=torch.int32))

        if i%100000==0:
            print(f"{i} satır işlendi, token sayısı: {len(tokens_chunk)}")
token_tensor= torch.cat(tokens_chunk)
torch.save(token_tensor,"/workspaces/llm-from-scratch/data/wikipedia_encoded.pt")
print("Tokenizasyon tamamlandı, toplam token:", len(token_tensor))


import torch
from torch.utils.data import random_split, DataLoader
from türkçe_dil_modelim.text_dataset import TextDataset


context_length = 256  # önceki 128 yerine biraz daha uzun, LLM için daha iyi
stride = 256            # önceki 32 yerine artırdık, overlap azalır → RAM dostu
batch_size = 16        # M1 için güvenli
shuffle_train = True   # train loader shuffle, val/test shuffle yok


token_tensor = torch.load(
    "/Users/burak.yilmaz/Desktop/kendi_dil_modelim/encoded_wikipedia_150m_24k_vocab.pt"
)
print("Toplam token:", len(token_tensor))


full_dataset = TextDataset(
    token_tensor,
    context_length=context_length,
    stride=stride
)
print("Toplam sample:", len(full_dataset))



train_size = int(0.9 * len(full_dataset))
val_size = int(0.05 * len(full_dataset))
test_size = len(full_dataset) - train_size - val_size

train_dataset, val_dataset, test_dataset = random_split(
    full_dataset,
    [train_size, val_size, test_size]
)

print("Train samples:", len(train_dataset))
print("Val samples:", len(val_dataset))
print("Test samples:", len(test_dataset))



train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=shuffle_train,
    num_workers=4,
    pin_memory=False, # M1'de pin_memory genelde performansı düşürür, bu yüzden False yaptım
    persistent_workers=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=4,
    pin_memory=False,
    persistent_workers=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=4,
    pin_memory=False,
    persistent_workers=True
)
device = "mps" if torch.backends.mps.is_available() else "cpu"
model = Model(
    vocab_size=len(tokenizer.vocab),
    embedding_dim=384,
    num_heads=12,
    num_layers=8,
    context_length=context_length,
    device=device
)
print("Model parametre sayısı:", sum(p.numel() for p in model.parameters()))
print("Device:", device)

trainer = Trainer(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    test_loader=test_loader,
    device=device,
    lr=1e-4,
    weight_decay=0.01,
    epochs=10,
    warmup_steps=10000,
    grad_clip=0.5,
    save_dir="training_outputs",
    early_stopping_patience=3
)



trainer.train()